# E2 - YOLO26n DetectionHead Multi-Source Domain Generalization

This experiment asks one focused question: can the lightweight YOLO26n DetectionHead retain edge suitability while becoming more reliable on a construction dataset that was not used for training?

E2 keeps the four project classes: `helmet`, `no-helmet`, `vest`, and `no-vest`. It changes the training recipe, not the runtime task. The experiment uses MuSGD, source-balanced sampling, moderate camera variation, and leave-one-source-out evaluation.

## 1. Research design

The detector is trained on complete-label sources only. Sources that annotate helmet but omit vest must not be treated as negative vest evidence. One complete source is held out from training and used only after model selection. This measures cross-source generalization more honestly than a random image split.

Primary metrics are held-out mAP50-95, `no-helmet` recall, `no-vest` recall, and the worst-source mAP50-95. Edge cost is measured later on the target Jetson with TensorRT FP16.

References: [YOLO26 training recipe](https://docs.ultralytics.com/guides/yolo26-training-recipe), [YOLO26 architecture](https://docs.ultralytics.com/models/yolo26), and [Domain-Generalized Object Detection, CVPR 2024](https://openaccess.thecvf.com/content/CVPR2024/html/Danish_Improving_Single_Domain-Generalized_Object_Detection_A_Focus_on_Diversification_and_CVPR_2024_paper.html).

In [ ]:
import importlib.util
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from packaging.version import Version

packages = []
try:
    if Version(version("ultralytics")) < Version("8.4.190"):
        packages.append("ultralytics>=8.4.190")
except PackageNotFoundError:
    packages.append("ultralytics>=8.4.190")
if importlib.util.find_spec("albumentations") is None:
    packages.append("albumentations>=1.4.0")
if packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *packages])
print("Environment ready.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import json
import math
import platform
import random
import shutil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import yaml
from IPython.display import display
from ultralytics import YOLO
import ultralytics

DATASET_ROOT_OVERRIDE = None
OUTPUT_ROOT_OVERRIDE = None
RESUME_WEIGHTS = None  # Set to a previous E2 last.pt in a later Kaggle Version.
PREVIOUS_BEST_WEIGHTS = None
HELD_OUT_SOURCE = "css"  # Choose: tucje, gsnvb, mendeley, css. Use None only for the final all-source model.
COMPLETE_SOURCES = ["tucje", "gsnvb", "mendeley", "css"]
PARTIAL_SOURCES = ["legacy", "shwd", "chv", "pictor"]
CLASS_NAMES = {0: "helmet", 1: "no-helmet", 2: "vest", 3: "no-vest"}
RUN_TRAINING = True
RUN_EXPORT = False
EPOCHS = 100
EPOCH_IMAGES = 24000
SOURCE_BALANCE_ALPHA = 0.5
IMAGE_SIZE = 640
SEED = 42
SAVE_PERIOD = 5

GPU_COUNT = torch.cuda.device_count()
USE_DUAL_GPU = GPU_COUNT >= 2
TRAIN_DEVICE = [0, 1] if USE_DUAL_GPU else (0 if GPU_COUNT == 1 else "cpu")
EVAL_DEVICE = 0 if GPU_COUNT >= 1 else "cpu"
BATCH = 32 if USE_DUAL_GPU else 16
WORKERS = 2 if USE_DUAL_GPU else 4

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
if HELD_OUT_SOURCE is not None and HELD_OUT_SOURCE not in COMPLETE_SOURCES:
    raise ValueError(f"HELD_OUT_SOURCE must be one of {COMPLETE_SOURCES} or None")
print("Ultralytics:", ultralytics.__version__)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(GPU_COUNT)])
print("Training device:", TRAIN_DEVICE, "| batch:", BATCH, "| held out:", HELD_OUT_SOURCE)

In [ ]:
def is_dataset_root(path):
    val_name = "valid" if (path / "images" / "valid").exists() else "val"
    return (path / "images" / "train").exists() and (path / "images" / val_name).exists()

def find_dataset_root():
    if DATASET_ROOT_OVERRIDE:
        candidate = Path(DATASET_ROOT_OVERRIDE).expanduser()
        if is_dataset_root(candidate):
            return candidate.resolve()
        raise FileNotFoundError(f"Invalid DATASET_ROOT_OVERRIDE: {candidate}")
    candidates = [
        Path("/kaggle/input/datasets/linhlaz225/helmet-nohelmet-vest-novest/Dataset_v2"),
        Path.cwd() / "Dataset_v2",
        Path.cwd().parent / "Dataset" / "Dataset_v2",
    ]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        candidates.extend(p for p in kaggle_input.rglob("Dataset_v2") if p.is_dir())
    for candidate in candidates:
        if candidate.exists() and is_dataset_root(candidate):
            return candidate.resolve()
    raise FileNotFoundError("Dataset V2 was not found. Set DATASET_ROOT_OVERRIDE.")

DATASET_ROOT = find_dataset_root()
VAL_FOLDER = "valid" if (DATASET_ROOT / "images" / "valid").exists() else "val"
OUTPUT_ROOT = Path(OUTPUT_ROOT_OVERRIDE).resolve() if OUTPUT_ROOT_OVERRIDE else (
    Path("/kaggle/working/ppe_e2_dg_study") if Path("/kaggle/working").exists()
    else Path.cwd() / "ai-model" / "outputs" / "ppe_e2_dg_study"
)
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
CONFIG_DIR = OUTPUT_ROOT / "config"
RUNS_DIR = OUTPUT_ROOT / "runs"
RESULTS_DIR = OUTPUT_ROOT / "results"
RECOVERY_DIR = OUTPUT_ROOT / "recovery"
EXPORT_DIR = OUTPUT_ROOT / "export"
for folder in [MANIFEST_DIR, CONFIG_DIR, RUNS_DIR, RESULTS_DIR, RECOVERY_DIR, EXPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
print("Dataset:", DATASET_ROOT)
print("Output:", OUTPUT_ROOT)

## 2. Source and label audit

The filename prefix identifies the source. This audit verifies that every selected image has a valid four-class YOLO label and shows why source balancing is needed.

In [ ]:
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def source_name(path):
    return path.stem.split("__", 1)[0].lower()

def split_images(split):
    folder = DATASET_ROOT / "images" / split
    return sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES)

def label_path(image_path):
    split = image_path.parent.name
    return DATASET_ROOT / "labels" / split / f"{image_path.stem}.txt"

def label_counts(image_path):
    counts = Counter()
    path = label_path(image_path)
    if not path.exists():
        raise FileNotFoundError(f"Missing label: {path}")
    for line_number, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
        if not line.strip():
            continue
        parts = line.split()
        if len(parts) != 5:
            raise ValueError(f"Invalid YOLO row at {path}:{line_number}")
        class_id = int(float(parts[0]))
        coordinates = [float(value) for value in parts[1:]]
        if class_id not in CLASS_NAMES or not all(0 <= value <= 1 for value in coordinates):
            raise ValueError(f"Invalid label at {path}:{line_number}")
        counts[class_id] += 1
    return counts

audit_rows = []
for split in ["train", VAL_FOLDER, "test"]:
    for image_path in split_images(split):
        source = source_name(image_path)
        counts = label_counts(image_path)
        row = {"split": split, "source": source, "image": str(image_path), "boxes": sum(counts.values())}
        row.update({CLASS_NAMES[class_id]: counts[class_id] for class_id in CLASS_NAMES})
        audit_rows.append(row)
audit_df = pd.DataFrame(audit_rows)
source_table = audit_df.groupby(["source", "split"]).agg(
    images=("image", "count"), boxes=("boxes", "sum"),
    helmet=("helmet", "sum"), no_helmet=("no-helmet", "sum"),
    vest=("vest", "sum"), no_vest=("no-vest", "sum"),
).reset_index()
display(source_table)
source_table.to_csv(RESULTS_DIR / "source_audit.csv", index=False)

train_counts = audit_df[audit_df["split"] == "train"].groupby("source").size().sort_values(ascending=False)
ax = train_counts.plot(kind="bar", figsize=(10, 4), color="#111111", title="Training images by source before balancing")
ax.set_ylabel("Images")
plt.tight_layout()
plt.show()

## 3. Build the domain-balanced training split

Sampling probability is proportional to the square root of source size. Large sources keep more samples, while small sources receive enough representation without being repeated as aggressively as strict equal balancing. Partial-label sources are excluded from the core E2 loss.

In [ ]:
rng = random.Random(SEED)
training_sources = [source for source in COMPLETE_SOURCES if source != HELD_OUT_SOURCE]
train_pool = {
    source: [Path(value) for value in audit_df[(audit_df["split"] == "train") & (audit_df["source"] == source)]["image"].tolist()]
    for source in training_sources
}
if any(not images for images in train_pool.values()):
    raise RuntimeError(f"A complete source has no training images: { {k: len(v) for k, v in train_pool.items()} }")
unique_training_images = sum(len(images) for images in train_pool.values())
effective_epoch_images = min(EPOCH_IMAGES, unique_training_images * 4)
weights = {source: len(images) ** SOURCE_BALANCE_ALPHA for source, images in train_pool.items()}
weight_sum = sum(weights.values())
targets = {source: max(1, round(effective_epoch_images * weight / weight_sum)) for source, weight in weights.items()}
targets[training_sources[-1]] += effective_epoch_images - sum(targets.values())

balanced_images = []
for source, images in train_pool.items():
    target = targets[source]
    selected = rng.sample(images, target) if target <= len(images) else [rng.choice(images) for _ in range(target)]
    balanced_images.extend(selected)
rng.shuffle(balanced_images)

validation_images = [
    Path(value) for value in audit_df[(audit_df["split"] == VAL_FOLDER) & audit_df["source"].isin(training_sources)]["image"].tolist()
]
heldout_images = []
if HELD_OUT_SOURCE is not None:
    heldout_images = [
        Path(value) for value in audit_df[audit_df["source"].eq(HELD_OUT_SOURCE) & audit_df["split"].isin([VAL_FOLDER, "test"])]["image"].tolist()
    ]
    if not heldout_images:
        raise RuntimeError(f"No held-out evaluation images for {HELD_OUT_SOURCE}")

def write_manifest(name, images):
    path = MANIFEST_DIR / name
    path.write_text("\n".join(str(image.resolve()) for image in images) + "\n", encoding="utf-8")
    return path

TRAIN_MANIFEST = write_manifest("train_balanced.txt", balanced_images)
VAL_MANIFEST = write_manifest("val_training_sources.txt", validation_images)
HELDOUT_MANIFEST = write_manifest(f"heldout_{HELD_OUT_SOURCE}.txt", heldout_images) if heldout_images else None
balance_df = pd.DataFrame({
    "source": training_sources,
    "available_train": [len(train_pool[source]) for source in training_sources],
    "sampled_per_epoch": [targets[source] for source in training_sources],
})
display(balance_df)
balance_df.to_csv(RESULTS_DIR / "source_balance.csv", index=False)
print("Training images per epoch:", len(balanced_images))
print("Training-source validation images:", len(validation_images))
print("Held-out images:", len(heldout_images))

In [ ]:
data_config = {
    "path": str(DATASET_ROOT),
    "train": str(TRAIN_MANIFEST),
    "val": str(VAL_MANIFEST),
    "test": str(HELDOUT_MANIFEST or VAL_MANIFEST),
    "nc": 4,
    "names": CLASS_NAMES,
}
DATA_YAML = CONFIG_DIR / "dataset_e2.yaml"
DATA_YAML.write_text(yaml.safe_dump(data_config, sort_keys=False), encoding="utf-8")
print(DATA_YAML.read_text(encoding="utf-8"))

## 4. E2 model and training recipe

YOLO26n DetectionHead predicts at strides 4, 8, 16, and 32. The stride-4 head layers are initialized after transferring compatible weights from `yolo26n.pt`. MuSGD follows the official YOLO26 direction. Moderate geometric changes preserve PPE shape, while scale, HSV, mosaic, MixUp, and Ultralytics' light blur/CLAHE transforms provide camera variation.

In [ ]:
RUN_NAME = f"E2_detectionhead_musgd_{'loso_' + HELD_OUT_SOURCE if HELD_OUT_SOURCE else 'all_sources'}"
RUN_DIR = RUNS_DIR / RUN_NAME
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
LAST_WEIGHTS = RUN_DIR / "weights" / "last.pt"
RESULTS_CSV = RUN_DIR / "results.csv"

def find_pretrained_yolo26n():
    candidates = [
        Path("/kaggle/input/models/quanghuy225/yolo-p2-resume/pytorch/default/1/yolo26n.pt"),
        Path("/kaggle/input/models/quanghuy225/yolo-p2-resume/pytorch/default/1/weights/yolo26n.pt"),
        Path.cwd() / "yolo26n.pt",
    ]
    model_root = Path("/kaggle/input/models/quanghuy225/yolo-p2-resume/pytorch/default/1")
    if model_root.exists():
        candidates.extend(model_root.rglob("yolo26n.pt"))
    for candidate in candidates:
        if candidate.exists():
            return str(candidate.resolve())
    return "yolo26n.pt"

PRETRAINED_WEIGHTS = find_pretrained_yolo26n()
print("Pretrained source:", PRETRAINED_WEIGHTS)

def completed_epochs():
    if RESULTS_CSV.exists():
        try:
            return len(pd.read_csv(RESULTS_CSV))
        except Exception:
            return 0
    return 0

def stage_resume_files(source_last):
    source_last = Path(source_last).expanduser().resolve()
    if not source_last.exists():
        raise FileNotFoundError(f"Resume checkpoint not found: {source_last}")
    LAST_WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
    if source_last != LAST_WEIGHTS.resolve():
        shutil.copy2(source_last, LAST_WEIGHTS)
    best_candidates = []
    if PREVIOUS_BEST_WEIGHTS:
        best_candidates.append(Path(PREVIOUS_BEST_WEIGHTS).expanduser())
    best_candidates.append(source_last.with_name("best.pt"))
    for candidate in best_candidates:
        if candidate.exists():
            shutil.copy2(candidate, BEST_WEIGHTS)
            break
    for candidate in [source_last.parent.parent / "results.csv", source_last.parent / "results.csv"]:
        if candidate.exists():
            shutil.copy2(candidate, RESULTS_CSV)
            break
    return LAST_WEIGHTS

def save_recovery(trainer):
    RECOVERY_DIR.mkdir(parents=True, exist_ok=True)
    for source in [Path(trainer.last), Path(trainer.best), Path(trainer.csv), Path(trainer.save_dir) / "args.yaml"]:
        if source.exists():
            shutil.copy2(source, RECOVERY_DIR / source.name)
    state = {
        "experiment": RUN_NAME, "completed_epochs": int(trainer.epoch) + 1,
        "held_out_source": HELD_OUT_SOURCE, "updated_utc": datetime.now(timezone.utc).isoformat(),
    }
    (RECOVERY_DIR / "state.json").write_text(json.dumps(state, indent=2), encoding="utf-8")

In [ ]:
train_args = {
    "data": str(DATA_YAML), "epochs": EPOCHS, "imgsz": IMAGE_SIZE, "batch": BATCH,
    "workers": WORKERS, "device": TRAIN_DEVICE, "seed": SEED, "deterministic": True,
    "optimizer": "MuSGD", "lr0": 0.0054, "lrf": 0.0495, "momentum": 0.947,
    "weight_decay": 0.00064, "warmup_epochs": 1.0, "cos_lr": True,
    "hsv_h": 0.015, "hsv_s": 0.60, "hsv_v": 0.45, "degrees": 3.0,
    "translate": 0.10, "scale": 0.50, "fliplr": 0.5, "mosaic": 0.70,
    "mixup": 0.05, "multi_scale": 0.15, "close_mosaic": 10, "amp": True, "nbs": 64,
    "project": str(RUNS_DIR), "name": RUN_NAME, "exist_ok": True,
    "save": True, "save_period": SAVE_PERIOD, "val": True, "patience": 35, "plots": True,
}
manifest = {
    "experiment": RUN_NAME, "architecture": "yolo26n-p2.yaml",
    "pretrained_transfer": PRETRAINED_WEIGHTS, "classes": CLASS_NAMES,
    "complete_sources": COMPLETE_SOURCES, "partial_sources_excluded": PARTIAL_SOURCES,
    "training_sources": training_sources, "held_out_source": HELD_OUT_SOURCE,
    "source_balance_alpha": SOURCE_BALANCE_ALPHA, "images_per_epoch": len(balanced_images),
    "training": train_args,
    "environment": {"python": platform.python_version(), "torch": torch.__version__, "ultralytics": ultralytics.__version__},
}
(RESULTS_DIR / "experiment_manifest.json").write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")
display(pd.DataFrame([train_args]).T.rename(columns={0: "value"}))

if RUN_TRAINING:
    resume_path = Path(RESUME_WEIGHTS).expanduser() if RESUME_WEIGHTS else LAST_WEIGHTS
    if completed_epochs() >= EPOCHS and BEST_WEIGHTS.exists():
        print(f"E2 already complete: {completed_epochs()}/{EPOCHS} epochs")
    elif resume_path.exists():
        resume_path = stage_resume_files(resume_path)
        print("Resuming E2 from:", resume_path)
        model = YOLO(str(resume_path))
        model.add_callback("on_model_save", save_recovery)
        model.train(resume=True, data=str(DATA_YAML), device=TRAIN_DEVICE, workers=WORKERS, batch=BATCH, save_period=SAVE_PERIOD, plots=True)
    else:
        model = YOLO("yolo26n-p2.yaml")
        model.load(PRETRAINED_WEIGHTS)
        model.add_callback("on_model_save", save_recovery)
        model.train(**train_args)
else:
    print("Training disabled.")

## 5. Pooled and source-specific evaluation

The model is selected using validation images from training sources only. The held-out source is evaluated afterward. The source table exposes average performance and the weakest domain instead of hiding them inside one pooled number.

In [ ]:
ACTIVE_BEST = BEST_WEIGHTS if BEST_WEIGHTS.exists() else RECOVERY_DIR / "best.pt"
if not ACTIVE_BEST.exists():
    raise FileNotFoundError("E2 best.pt was not found. Complete training or attach the previous E2 output.")

def metric_record(metrics, source, role, image_count):
    box = metrics.box
    record = {
        "source": source, "role": role, "images": image_count,
        "precision_macro": float(box.mp), "recall_macro": float(box.mr),
        "map50": float(box.map50), "map50_95": float(box.map),
    }
    class_ids = [int(value) for value in np.asarray(box.ap_class_index).tolist()]
    for index, class_id in enumerate(class_ids):
        name = CLASS_NAMES[class_id]
        record[f"{name}_ap50_95"] = float(np.asarray(box.ap)[index])
        record[f"{name}_recall"] = float(np.asarray(box.r)[index])
    return record

def evaluate_manifest(model, source, role, images):
    manifest_path = write_manifest(f"eval_{source}_{role}.txt", images)
    config = {"path": str(DATASET_ROOT), "train": str(TRAIN_MANIFEST), "val": str(manifest_path), "nc": 4, "names": CLASS_NAMES}
    yaml_path = CONFIG_DIR / f"eval_{source}_{role}.yaml"
    yaml_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
    metrics = model.val(data=str(yaml_path), split="val", imgsz=IMAGE_SIZE, batch=16, device=EVAL_DEVICE, plots=False, verbose=False)
    return metric_record(metrics, source, role, len(images))

best_model = YOLO(str(ACTIVE_BEST))
evaluation_rows = []
evaluation_rows.append(evaluate_manifest(best_model, "pooled_training_sources", "validation", validation_images))
for source in COMPLETE_SOURCES:
    images = [
        Path(value) for value in audit_df[audit_df["source"].eq(source) & audit_df["split"].isin([VAL_FOLDER, "test"])]["image"].tolist()
    ]
    if images:
        role = "held_out" if source == HELD_OUT_SOURCE else "seen_source"
        evaluation_rows.append(evaluate_manifest(best_model, source, role, images))
evaluation_df = pd.DataFrame(evaluation_rows)
evaluation_df.to_csv(RESULTS_DIR / "source_generalization_metrics.csv", index=False)
display(evaluation_df)

source_only = evaluation_df[evaluation_df["source"].isin(COMPLETE_SOURCES)]
summary = {
    "mean_source_map50_95": float(source_only["map50_95"].mean()),
    "worst_source_map50_95": float(source_only["map50_95"].min()),
    "worst_source": source_only.loc[source_only["map50_95"].idxmin(), "source"],
    "held_out_source": HELD_OUT_SOURCE,
}
if HELD_OUT_SOURCE is not None:
    heldout_row = evaluation_df[evaluation_df["source"] == HELD_OUT_SOURCE].iloc[0]
    summary.update({
        "held_out_map50_95": float(heldout_row["map50_95"]),
        "held_out_no_helmet_recall": float(heldout_row.get("no-helmet_recall", np.nan)),
        "held_out_no_vest_recall": float(heldout_row.get("no-vest_recall", np.nan)),
    })
(RESULTS_DIR / "generalization_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
display(pd.DataFrame([summary]).T.rename(columns={0: "value"}))

plt.figure(figsize=(8, 4))
sns.barplot(data=source_only, x="source", y="map50_95", color="#202020")
plt.ylim(0, 1)
plt.title("E2 mAP50-95 by source")
plt.tight_layout()
plt.show()

In [ ]:
if RUN_EXPORT:
    exported = best_model.export(format="onnx", imgsz=IMAGE_SIZE, dynamic=False, simplify=True, opset=17)
    exported = Path(exported)
    target = EXPORT_DIR / "ppe_e2_yolo26n_p2_fp32.onnx"
    shutil.copy2(exported, target)
    print("ONNX export:", target)
else:
    print("Export disabled. Build TensorRT FP16 on the target Jetson after E2 is selected.")

## 6. Decision rule

Run one Kaggle Version for each held-out complete source. E2 is preferred only when it improves mean and worst-source mAP50-95 or violation recall without an unacceptable Jetson latency increase. After the leave-one-source-out study, set `HELD_OUT_SOURCE = None` and train one final deployment model on all four complete sources. Do not add partial-label sources unless the loss can mask classes that were not annotated.